In [4]:
pip install scikit-learn

  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.4 MB 5.6 MB/s eta 0:00:02
   ------ --------------------------------- 1.3/8.4 MB 5.5 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/8.4 MB 3.4 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.4 MB 3.1 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.4 MB 3.0 MB/s eta 0:00:02
   ------------------ --------------------- 3.9/8.4 MB 3.1 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.4 MB 2.9 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.4 MB 2.7 MB/s eta 0:00:02
   ---------------------- ----------------- 4.7/8.4 MB 2.6 MB/s eta 0:00:02
   ----------------------- ---------------- 5.0/8.4 MB 2.5 MB/s eta 0:00:02
   -------------------------- ------------- 5.5/8.4 MB 2.4 MB/s eta 0:00:02
   ---------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install scikit-learn xgboost shap imbalanced-learn


  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 0.8/48.9 MB 5.3 MB/s eta 0:00:10
   - -------------------------------------- 1.8/48.9 MB 5.2 MB/s eta 0:00:10
   - -------------------------------------- 2.4/48.9 MB 4.1 MB/s eta 0:00:12
   -- ------------------------------------- 2.6/48.9 MB 4.0 MB/s eta 0:00:12
   -- ------------------------------------- 2.9/48.9 MB 2.8 MB/s eta 0:00:17
   -- ------------------------------------- 3.1/48.9 MB 2.7 MB/s eta 0:00:18
   -- ------------------------------------- 3.7/48.9 MB 2.5 MB/s eta 0:00:19
   -- ------------------------------------- 3.7/48.9 MB 2.5 MB/s eta 0:00:19
   -- ------------------------------------- 3.7/48.9 MB 2.5 MB/s eta 0:00:19
   -- ------------------------------------- 3.7/48.9 MB 2.5 MB/s eta 0:00:19
   -- ------------------------------------- 3.7/48.9 MB 2.5 MB/s eta 0:00:19
   -- ----------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

In [23]:
df = pd.read_csv('../data/processed/cleaned_data.csv')

print("Cleaned data loaded")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Cleaned data loaded
Shape: (938599, 41)
Columns: ['id', 'spkid', 'full_name', 'pdes', 'neo', 'pha', 'H', 'diameter', 'orbit_id', 'epoch', 'epoch_mjd', 'epoch_cal', 'equinox', 'e', 'a', 'q', 'i', 'om', 'w', 'ma', 'ad', 'n', 'tp', 'tp_cal', 'per', 'per_y', 'moid', 'moid_ld', 'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w', 'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per', 'class', 'rms']


C:\Users\surek\AppData\Local\Temp\ipykernel_15048\2615915532.py:1: DtypeWarning: Columns (3: pdes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/cleaned_data.csv')


In [24]:
#Drop Useless Columns
# Identifiers - no prediction value
id_cols = ['id', 'spkid', 'pdes', 'full_name', 'orbit_id', 'equinox']

# Duplicate / derived columns
dup_cols = ['epoch', 'epoch_mjd', 'epoch_cal', 'tp', 'tp_cal',
            'per', 'per_y', 'ad', 'moid_ld', 
            'sigma_per', 'sigma_ad', 'sigma_i', 'sigma_q',
            'sigma_om', 'sigma_ma', 'sigma_n', 'sigma_tp',
            'sigma_e', 'sigma_a', 'sigma_w', 'rms']

drop_cols = [c for c in id_cols + dup_cols if c in df.columns]

df = df.drop(columns=drop_cols)

print("Dropped columns:", drop_cols)
print("Shape after drop:", df.shape)
print("Remaining:", df.columns.tolist())

Dropped columns: ['id', 'spkid', 'pdes', 'full_name', 'orbit_id', 'equinox', 'epoch', 'epoch_mjd', 'epoch_cal', 'tp', 'tp_cal', 'per', 'per_y', 'ad', 'moid_ld', 'sigma_per', 'sigma_ad', 'sigma_i', 'sigma_q', 'sigma_om', 'sigma_ma', 'sigma_n', 'sigma_tp', 'sigma_e', 'sigma_a', 'sigma_w', 'rms']
Shape after drop: (938599, 14)
Remaining: ['neo', 'pha', 'H', 'diameter', 'e', 'a', 'q', 'i', 'om', 'w', 'ma', 'n', 'moid', 'class']


In [25]:
#Check Remaining Columns
print("Remaining columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

Remaining columns:
1. neo
2. pha
3. H
4. diameter
5. e
6. a
7. q
8. i
9. om
10. w
11. ma
12. n
13. moid
14. class


In [26]:
#Handle Skewness — Check
# Skewed features - apply log transform
skewed_cols = ['a', 'q', 'n', 'i', 'moid', 'e', 'H']

for col in skewed_cols:
    if col in df.columns:
        df[col + '_log'] = np.log1p(df[col].abs())
        df = df.drop(columns=[col])

# diameter-க்கு log version create, original-ஐயும் வை (target)
if 'diameter' in df.columns:
    df['diameter_log'] = np.log1p(df['diameter'])

print("Log transform applied")
print("Remaining columns:", df.columns.tolist())

Log transform applied
Remaining columns: ['neo', 'pha', 'diameter', 'om', 'w', 'ma', 'class', 'a_log', 'q_log', 'n_log', 'i_log', 'moid_log', 'e_log', 'H_log', 'diameter_log']


In [21]:
#Log Transform for Skewed Features
# Skewed columns
skewed_cols = ['a', 'q', 'n', 'i', 'moid', 'diameter', 'e', 'H']

# Apply log and drop original
for col in skewed_cols:
    if col in df.columns:
        df[col + '_log'] = np.log1p(df[col].abs())
        df = df.drop(columns=[col])

print("Log transform applied, originals dropped")
print("Remaining columns:", df.columns.tolist())

Log transform applied, originals dropped
Remaining columns: ['om', 'w', 'ma', 'a_log', 'q_log', 'n_log', 'i_log', 'moid_log', 'diameter_log', 'e_log', 'H_log', 'neo_encoded', 'pha_encoded', 'orbit_id_encoded', 'equinox_encoded', 'class_encoded']


In [27]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))

# Drop original categorical columns
df = df.drop(columns=cat_cols)

print("Encoding done")
print("Shape:", df.shape)


C:\Users\surek\AppData\Local\Temp\ipykernel_15048\717125006.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


Categorical columns: ['neo', 'pha', 'class']
Encoding done
Shape: (938599, 15)


In [29]:

print("PREPROCESSED DATA SUMMARY")

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")


PREPROCESSED DATA SUMMARY
Total rows: 938599
Total columns: 15
Missing values: 0

Columns:
1. diameter
2. om
3. w
4. ma
5. a_log
6. q_log
7. n_log
8. i_log
9. moid_log
10. e_log
11. H_log
12. diameter_log
13. neo_encoded
14. pha_encoded
15. class_encoded


In [30]:
df.to_csv('../data/processed/preprocessed_data.csv', index=False)

print("Preprocessed data saved")
print("Location: ../data/processed/preprocessed_data.csv")
print("Shape:", df.shape)

Preprocessed data saved
Location: ../data/processed/preprocessed_data.csv
Shape: (938599, 15)
